## Module 5: Deploying ML Models with FastAPI and uv

This module focuses on **deploying machine learning models** and making them accessible to other applications through web services. Building on the classification model from Modules 3 and 4, we move from training and evaluating models in a notebook to **saving a model, serving predictions through an API, and packaging the application**.

The main topics include **pickle (saving and loading models)**, **DictVectorizer and Scikit-Learn Pipelines**, **FastAPI and Pydantic**, **HTTP requests**, **uv for reproducible Python environments**, and **Docker**. We use the same **Telco customer churn** example as the *updated Module 5 uv/FastAPI workshop*; the 2026 lead-scoring homework uses a separate pretrained model.

### Part 1: Model Preparation (Modules 3 and 4 Recap)

We use the [Telco customer churn dataset](https://www.kaggle.com/datasets/blastchar/telco-customer-churn). The focus here is **deployment**, so we train one logistic regression model on the full dataset instead of repeating the validation experiments from Module 4.

In [ ]:
# import used libraries
import pickle

import numpy as np
import pandas as pd
import sklearn

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# check library versions (helpful when saving/loading models)
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')
print(f'sklearn: {sklearn.__version__}')

In [ ]:
# get telecom churn data
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_url)

# rewrite column names lowercase with no spaces
df.columns = df.columns.str.lower().str.replace(' ', '_')

# make categorical values consistent
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)
for col in categorical_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')

# totalcharges sometimes contains blank values
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce').fillna(0)

# change target from yes/no to 1/0
df.churn = (df.churn == 'yes').astype(int)
df.head()

In [ ]:
# separate target variable
y_train = df.churn

numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents', 'phoneservice',
    'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup',
    'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies',
    'contract', 'paperlessbilling', 'paymentmethod'
]

# the vectorizer converts dictionaries into numerical features
train_dict = df[categorical + numerical].to_dict(orient='records')
dv = DictVectorizer()  # sparse representation, as in the updated workshop
X_train = dv.fit_transform(train_dict)

model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)
print(f'Trained on {X_train.shape[0]} customers and {X_train.shape[1]} encoded features')

### Part 2: Predicting for One Customer

Like in Module 3, we start with a **dictionary containing one customer's features**. With separate objects, we first call `dv.transform()` and then `model.predict_proba()`. The `[0, 1]` picks the first customer's **positive-class probability** (churn).

In [ ]:
# customer from the updated workshop
datapoint = {
    'gender': 'female', 'seniorcitizen': 0, 'partner': 'yes',
    'dependents': 'no', 'phoneservice': 'no',
    'multiplelines': 'no_phone_service', 'internetservice': 'dsl',
    'onlinesecurity': 'no', 'onlinebackup': 'yes',
    'deviceprotection': 'no', 'techsupport': 'no',
    'streamingtv': 'no', 'streamingmovies': 'no',
    'contract': 'month-to-month', 'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check', 'tenure': 1,
    'monthlycharges': 29.85, 'totalcharges': 29.85
}

# transform the dictionary before predicting
X = dv.transform([datapoint])
churn_probability = model.predict_proba(X)[0, 1]
print(f'Churn probability: {churn_probability:.3f}')

### Part 3: Saving and Loading a Model with Pickle

**Pickle** saves Python objects in binary form so that we do not need to retrain every time the prediction service starts.

When the vectorizer and classifier are **separate objects**, save both in a tuple `(dv, model)` and unpack them together after loading. Pickle files should only be loaded from **trusted sources**. Model and library versions should match between training and deployment.

In [ ]:
# save the vectorizer and trained classifier together
with open('model.bin', 'wb') as f_out:
    pickle.dump((dv, model), f_out)

# simulate a separate process loading these saved objects
with open('model.bin', 'rb') as f_in:
    dv_loaded, model_loaded = pickle.load(f_in)

X_loaded = dv_loaded.transform([datapoint])
loaded_probability = model_loaded.predict_proba(X_loaded)[0, 1]
print(f'Loaded model probability: {loaded_probability:.3f}')

### Part 4: Combining Preprocessing and Prediction with a Pipeline

An alternative from the updated workshop is a **Scikit-Learn Pipeline**. It connects `DictVectorizer` and `LogisticRegression`, so there is no separate `dv.transform()` call at inference time.

**Difference:** save `(dv, model)` if you trained separate objects; save **one pipeline object** if you trained a pipeline. This was also the pattern used by the 2026 homework's `pipeline.bin`.

In [ ]:
# create and train the complete preprocessing + model pipeline
pipeline = make_pipeline(
    DictVectorizer(),
    LogisticRegression(solver='liblinear')
)
pipeline.fit(train_dict, y_train)

# dictionaries can be passed directly to the pipeline
pipeline_probability = pipeline.predict_proba([datapoint])[0, 1]
print(f'Pipeline probability: {pipeline_probability:.3f}')

# save and reload the complete pipeline as ONE object
with open('pipeline.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

with open('pipeline.bin', 'rb') as f_in:
    loaded_pipeline = pickle.load(f_in)

print(f'Reloaded pipeline probability: {loaded_pipeline.predict_proba([datapoint])[0, 1]:.3f}')

### Part 5: Turning the Model into a FastAPI Web Service

Until now, predictions have only worked within Python. A **web service** allows another program to send customer data over HTTP and receive a JSON prediction.

The new workshop uses **FastAPI** (instead of the older Flask lectures) and **Pydantic** for input validation. The server is normally written in a separate `predict.py` file, rather than started from a notebook.

- `POST /predict`: accepts a JSON customer and returns a probability and a Boolean.
- `GET /health`: reports whether the service is up.
- The classifier is loaded **once at startup**, not retrained per request.

Example *standalone* `predict.py` (adapt this to the exact response fields of your project):

```python
import pickle
from fastapi import FastAPI
from pydantic import BaseModel

with open('pipeline.bin', 'rb') as f:
    pipeline = pickle.load(f)

app = FastAPI()

class Customer(BaseModel):
    gender: str
    seniorcitizen: int
    partner: str
    dependents: str
    phoneservice: str
    multiplelines: str
    internetservice: str
    onlinesecurity: str
    onlinebackup: str
    deviceprotection: str
    techsupport: str
    streamingtv: str
    streamingmovies: str
    contract: str
    paperlessbilling: str
    paymentmethod: str
    tenure: int
    monthlycharges: float
    totalcharges: float

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/predict')
def predict(customer: Customer):
    probability = float(pipeline.predict_proba([customer.model_dump()])[0, 1])
    return {'churn_probability': probability, 'churn': probability >= 0.5}
```

Save this example as `predict.py` alongside the lecture-generated `pipeline.bin`, then run it with `uv run uvicorn predict:app --host 0.0.0.0 --port 9696` in an appropriately configured environment. **In the 2026 homework**, the supplied `predict.py` already implements the *lead-scoring* API, so we reused it without rewriting it.

### Part 6: Calling the API from Python

In the workshop, Alexey used the `requests` library to send customer data to a running API. This cell assumes the corresponding **churn API** is already running on port `9696`; it does *not* start a server. The request is optional and skips cleanly if no server is available.

In [ ]:
# optional: run after starting a compatible churn API in a separate terminal
# pip install requests  (if requests is missing in your notebook environment)
try:
    import requests
    response = requests.post('http://localhost:9696/predict', json=datapoint, timeout=3)
    response.raise_for_status()
    predictions = response.json()
    print(predictions)
except ImportError:
    print('Install requests to test a running API: pip install requests')
except requests.RequestException as exc:
    print(f'No compatible API response received: {exc}')

### Part 7: Managing the Environment with uv

**uv** manages the Python interpreter, project dependencies, and a local `.venv`. Keeping `pyproject.toml` and `uv.lock` makes deployment reproducible.

Run these commands in a **VS Code / WSL terminal** (not notebook Python cells):

```bash
uv --version
uv sync --locked             # install exactly the locked dependencies
uv run python smoke_test.py  # run the project's smoke test
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

The running server takes over that terminal. Use a **second terminal** to make requests. The 2026 homework pins Python **3.11.15** in `.python-version` and includes its own dependencies; the lecture's churn example is a separate project.

### Part 8: Packaging the Service with Docker

**Docker** packages the application, model artifact, Python runtime, and dependencies so that it behaves consistently across environments.

A Dockerfile generally defines the base image (`FROM`), copies files (`COPY`), installs dependencies (`RUN`), and specifies the server startup command (`CMD`). In the 2026 homework, we use the **provided** Dockerfile rather than writing one from scratch.

In the homework directory:

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

- `-t` names/tags the built image.
- The final `.` supplies the Docker build context.
- `-p 9696:9696` maps the local port to the container port.
- `--rm` removes the stopped container (not the Docker image).

You cannot run a local Uvicorn server and the mapped Docker container on the same host port **at the same time**. Stop one using `Ctrl+C` before starting the other.

### Part 9: Deployment and Summary

**Local notebook → saved artifact → FastAPI web service → Docker container → optional cloud hosting** is the workflow illustrated by this module. Hosting in the cloud makes the endpoint accessible to other systems, but the 2026 homework only requires a working local API and container.

Key takeaways:

1. **Separate vectorizer + classifier:** transform inputs manually, then predict; pickle both objects.
2. **Pipeline:** preprocessing and classification happen together; pickle one object.
3. **FastAPI + Pydantic:** expose validated JSON requests and prediction responses.
4. **uv + lockfile:** use compatible, reproducible Python dependencies.
5. **Docker:** run the same service and model predictably in a container.

**Homework distinction:** The lecture uses a *churn* dataset and predicts `churn`; the 2026 homework uses a **pretrained lead-scoring pipeline** and predicts `conversion`. See the separate homework checklist for the exact commands we ran.